# 4. HARD — odpowiedź prawie w czasie rzeczywistym, zdanie po zdaniu

Cel: przebudować `server/main.py` i `webgui/scripts.js`, aby użytkownik usłyszał pierwsze zdanie, zanim zakończy się generowanie całego tekstu. Istniejące endpointy `/api/tts/preview` i `/api/tts/files` mają nadal działać.

> To nie jest natywny streaming pojedynczych próbek z modelu. `TTS.synthesize()` zwraca gotowy waveform po zakończeniu syntezy przekazanego fragmentu. W tym zadaniu zmniejszamy odczuwalne opóźnienie przez osobne generowanie i przesyłanie kolejnych zdań.

## 1. Potwierdź ograniczenie API

Uruchom komórkę bez tworzenia obiektu `TTS`. Zwróć uwagę, że metoda przyjmuje `max_chunk_length`, `silence_duration` i `verbose`, ale zwraca jedną parę `(waveform, duration)`.

In [ ]:
import inspect
from supertonic import TTS

print(inspect.signature(TTS.synthesize))

## 2. Zaprojektuj kontrakt strumienia

Dodaj endpoint `POST /api/tts/stream`, który przyjmuje ten sam JSON co podgląd, ale odpowiada jako `application/x-ndjson`. Każda linia odpowiedzi jest osobnym JSON-em:

```json
{"type":"chunk","index":0,"text":"Pierwsze zdanie.","audio_base64":"UklGR..."}
{"type":"chunk","index":1,"text":"Czy drugie już słychać?","audio_base64":"UklGR..."}
{"type":"done","chunks":2}
```

Wymagania kontraktu:

- pierwsza linia ma dotrzeć po wygenerowaniu pierwszego zdania, nie całego tekstu,
- `.`, `?` i `!` kończą zdanie,
- każde `audio_base64` zawiera kompletny, poprawny fragment WAV,
- błąd po rozpoczęciu odpowiedzi jest zdarzeniem `{"type":"error", ...}`, bo kodu HTTP nie można już wtedy zmienić,
- pusta odpowiedź i tekst bez zdań mają zostać odrzucone przed wysłaniem nagłówków.

## 3. Zadanie serwerowe — `server/main.py`

Wprowadź zmiany małymi krokami:

1. Dodaj importy `base64`, `json`, `re`, `Iterator` oraz `StreamingResponse`.
2. Napisz `split_sentences(text: str) -> list[str]`. Na początek użyj granicy `(?<=[.!?])\s+`, usuń puste elementy i zachowaj znak kończący.
3. Dodaj testy pomocniczej funkcji dla kropki, pytajnika, wykrzyknika, wielu spacji i tekstu bez końcowej interpunkcji.
4. Napisz synchroniczny generator `stream_sentence_events(request) -> Iterator[bytes]`. Dla każdego zdania utwórz kopię requestu przez `request.model_copy(update={"text": sentence})`, wywołaj istniejące `synthesize_wav()`, zakoduj WAV przez `base64.b64encode(...)` i zwróć jedną linię NDJSON zakończoną `\n`.
5. Na końcu generatora zwróć zdarzenie `done`. Przechwyć błąd wewnątrz generatora i zwróć zdarzenie `error` bez ujawniania stosu ani ścieżek lokalnych.
6. Dodaj `POST /api/tts/stream` zwracający `StreamingResponse(..., media_type="application/x-ndjson")` oraz nagłówek `Cache-Control: no-cache`.
7. Nie usuwaj `_tts_lock`. Jeden model ONNX nadal ma być używany bez równoległego wejścia z kilku żądań.

Nie używaj `async def` do bezpośredniego uruchamiania blokującej syntezy. Synchroniczny generator może zostać obsłużony przez pulę wątków FastAPI/Starlette.

## 4. Zadanie frontendowe — `webgui/scripts.js`

Dodaj osobną akcję **Odtwarzaj zdaniami**:

1. Wyślij formularz do `/api/tts/stream`.
2. Czytaj `response.body` przez `getReader()` i `TextDecoder`. Buforuj niepełną ostatnią linię.
3. Dla zdarzenia `chunk` zamień Base64 na `ArrayBuffer`, zdekoduj WAV przez `AudioContext.decodeAudioData()` i dodaj wynik do kolejki.
4. Odtwarzaj kolejkę sekwencyjnie — następne zdanie nie może wejść na poprzednie.
5. Pokaż statusy: `Generowanie zdania 1…`, `Odtwarzanie zdania 1/…`, `Gotowe` i błąd.
6. Dodaj `AbortController` i przycisk zatrzymania. Po anulowaniu wyczyść reader, kolejkę i aktualne źródło audio.
7. Zachowaj dotychczasowe akcje podglądu i zapisu WAV.

Jeżeli potrzebujesz nowego przycisku lub pola statusu, zaktualizuj także `webgui/index.html`. Nie zapisuj fragmentów streamingowych w `generated_audio/`.

## 5. Test czasu pierwszego zdania

Po implementacji uruchom serwer z terminala w katalogu projektu:

```powershell
.\.venv\Scripts\python.exe .\server\main.py
```

Następna komórka mierzy czas nadejścia każdej linii. `chunk 0` powinien pojawić się przed zdarzeniem `done`.

In [ ]:
import json
import time
import urllib.request

payload = json.dumps({
    "text": "Pierwsze zdanie powinno nadejść szybko. Czy drugie dotarło później? Trzecie kończy test!",
    "voice": "F2",
    "language": "pl",
}).encode("utf-8")
request = urllib.request.Request(
    "http://127.0.0.1:8000/api/tts/stream",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)

started_at = time.perf_counter()
with urllib.request.urlopen(request, timeout=180) as response:
    for raw_line in response:
        event = json.loads(raw_line)
        print(f"{time.perf_counter() - started_at:7.2f} s | {event['type']} | {event.get('index', '-')}")

## 6. Kryteria akceptacji i polecenia kontrolne

W osobnym terminalu wykonaj:

```powershell
py -m py_compile .\server\main.py
node --check .\webgui\scripts.js
git diff --check
```

Rozwiązanie jest ukończone, gdy:

- testy `split_sentences()` przechodzą dla `.`, `?` i `!`,
- co najmniej dwa zdarzenia `chunk` docierają przed `done`,
- pierwszy fragment zaczyna się odtwarzać przed wygenerowaniem ostatniego,
- fragmenty są odtwarzane w poprawnej kolejności i bez nakładania,
- anulowanie zatrzymuje pobieranie i dźwięk,
- stare endpointy oraz zapis pełnego WAV nadal działają.

## 7. Dodatkowe opcje HARD+

Po ukończeniu wersji podstawowej przetestuj:

- mniejszą liczbę `total_steps` tylko dla pierwszego zdania, aby skrócić czas pierwszego dźwięku,
- różne `speed`, głosy i języki przy tym samym tekście,
- skróty takie jak `dr`, `prof.` i `itd.`, które utrudniają proste dzielenie regexem,
- długie zdanie bez interpunkcji oraz limity długości pojedynczego fragmentu,
- pomiar `time_to_first_chunk`, czasu każdego zdania i całego żądania,
- WebSocket z komunikatem `cancel` zamiast NDJSON,
- przesyłanie surowego PCM zamiast Base64 WAV, jeśli priorytetem stanie się mniejszy narzut.

Porównaj jakość i opóźnienie. Nie nazywaj rozwiązania pełnym realtime, dopóki model nie emituje audio w trakcie syntezy pojedynczego zdania.